# 06 — Model Evaluation & Threshold Optimization
**Project:** Transactional Fraud Detection Analysis  
**Objective:** Evaluate models on PR-AUC, Recall, Precision, and F1; optimize decision thresholds across [0.10 - 0.90]; plot confusion matrices; and demonstrate real-time risk scoring.



In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

PROJECT_ROOT = Path("..").resolve()
sys.path.append(str(PROJECT_ROOT))

from src.data_loader import DataLoader
from src.data_cleaning import DataCleaner
from src.feature_engineering import FeatureEngineer
from src.preprocessing import DataPreprocessor
from src.model_training import ModelTrainer
from src.model_evaluation import ModelEvaluator
from src.prediction import FraudRiskScorer

loader = DataLoader()
df_clean, _ = DataCleaner().clean_data(loader.load_data())

preprocessor = DataPreprocessor()
X_train_raw, X_test_raw, y_train, y_test = preprocessor.split_data(df_clean, target_col="Class")

fe = FeatureEngineer()
X_train_fe = fe.fit_transform(X_train_raw)
X_test_fe = fe.transform(X_test_raw)

X_train_scaled = preprocessor.fit_transform(X_train_fe)
X_test_scaled = preprocessor.transform(X_test_fe)

trainer = ModelTrainer()
trained_models = trainer.train_all_models(X_train_scaled, y_train)
evaluator = ModelEvaluator()



## 1. Classifier Benchmark Comparison
Benchmarking models by PR-AUC (Average Precision), Precision, Recall, and F1-Score.



In [ ]:
comp_df = evaluator.compare_models(trained_models, X_test_scaled, y_test, training_times=trainer.training_times)
comp_df



## 2. Precision-Recall and ROC Curves



In [ ]:
fig_curves = evaluator.plot_pr_and_roc_curves(trained_models, X_test_scaled, y_test)
plt.show()



## 3. Operational Threshold Optimization
Tuning decision boundary from 0.10 to 0.90 to balance missed fraud loss (FN) vs false alarms (FP).



In [ ]:
best_model_name = comp_df.iloc[0]["Model"]
best_model = trained_models[best_model_name]

thresh_df, opt_thresh, _ = evaluator.optimize_thresholds(best_model, X_test_scaled, y_test)
print(f"Optimal Operating Threshold: {opt_thresh:.2f}")
thresh_df



## 4. Confusion Matrix at Optimal Threshold



In [ ]:
best_eval = evaluator.evaluate_model(best_model, X_test_scaled, y_test, model_name=best_model_name, threshold=opt_thresh)
evaluator.plot_confusion_matrix(y_test, best_eval["y_pred"], model_name=f"{best_model_name} (Thresh={opt_thresh:.2f})")
plt.show()



## 5. Feature Importance



In [ ]:
df_imp, _ = evaluator.plot_feature_importance(best_model, list(X_train_scaled.columns), top_n=12)
df_imp.head(10)



## 6. Real-Time Risk Scoring Simulation



In [ ]:
scorer = FraudRiskScorer(threshold=opt_thresh)
sample_tx = {"Time": 10800.0, "Amount": 350.00}
for i in range(1, 29):
    sample_tx[f"V{i}"] = 0.0
sample_tx["V14"] = -6.50
sample_tx["V17"] = -4.80
sample_tx["V4"] = 4.20

score_res = scorer.score_transaction(sample_tx)
print("Risk Assessment Result:")
for k, v in score_res.items():
    print(f"  {k}: {v}")

